# 00 - bge-m3 임베딩 전 단계 추적

문서 1건과 질의 1건이 **문자열에서 검색 점수가 되기까지** 무슨 일이 일어나는지를
프레임워크 없이 한 단계씩 손으로 따라간다. 이슈
[#5](https://github.com/Bkankim/krsec-rag-lab/issues/5), 마일스톤 M0.

추적 경로:

`원문 → 토큰화 → forward → pooling(CLS) → L2 정규화 → NumPy 코사인 → top-3 회수`

## 규칙

- **프레임워크 금지.** LangChain/LlamaIndex 계열을 쓰지 않는다. `sentence-transformers`,
  `transformers`, `torch`, `numpy`만 쓴다. 추상화 뒤에 숨은 단계를 보는 것이 목적이므로
  편의 API도 최소한만 쓰고, 같은 결과를 손으로 재현해 대조한다.
- **원문 라이선스.** PLAN.md 원칙 6에 따라 재배포 조건이 확인되지 않은 KISA 공지 본문은
  리포·노트북 출력 어디에도 넣지 않는다. 아래 문서 10건은 **전부 이 노트북을 위해 지어낸
  합성 텍스트**다. CVE-ID와 제품명만 실제 공개 식별자를 참조하며, 본문 문장은 KISA·NVD를
  비롯한 어떤 원문에서도 복사하지 않았다. 사실 확인용으로 쓰지 말 것.
- **재현성.** 모델 revision 고정, 시드 고정, device는 CPU 고정. MPS/CUDA는 커널·축약 순서
  차이로 미세한 비결정성이 생기는데, 텍스트 11건이면 CPU로 충분하므로 결정성을 택한다.

## 실행

```bash
uv sync --group embed
uv run --no-sync jupyter lab   # 또는 nbconvert --execute
```

무거운 의존성(torch, sentence-transformers)은 `embed` 그룹에만 있다. CI는
`uv sync --locked --dev`만 돌리므로 이 노트북 스택은 CI에 설치되지 않는다.

## 1. 환경 고정

결과가 며칠 뒤에도 같은 값으로 재현되려면 (a) 모델 가중치 버전, (b) 난수 시드,
(c) 연산 device 세 가지가 동시에 고정돼야 한다. 모델 이름만 적고 revision을 비워 두면
허브에서 가중치가 갱신되는 순간 어제의 점수표가 재현되지 않는다.

In [1]:
import hashlib
import json
import platform
import random
import warnings
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import torch

MODEL_NAME = "BAAI/bge-m3"
# 허브의 main이 움직여도 같은 가중치를 받도록 커밋 해시로 고정한다.
MODEL_REVISION = "5617a9f61b028005a4858fdac845db406aefb181"
DEVICE = "cpu"
SEED = 20260919

# tqdm이 ipywidgets 부재를 경고하며 로컬 절대 경로를 출력에 남기는 것을 막는다.
warnings.filterwarnings("ignore", message="IProgress not found")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)


def repo_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "pyproject.toml").exists():
            return base
    raise RuntimeError("pyproject.toml을 찾지 못했다")


REPO_ROOT = repo_root()
FIXTURE_DIR = REPO_ROOT / "fixtures" / "embeddings_demo"

print("python  :", platform.python_version())
print("torch   :", torch.__version__)
print("numpy   :", np.__version__)
print("device  :", DEVICE)
print("seed    :", SEED)
print("fixture :", FIXTURE_DIR.relative_to(REPO_ROOT), "(리포 루트 기준)")

python  : 3.13.14
torch   : 2.14.0
numpy   : 2.5.3
device  : cpu
seed    : 20260919
fixture : fixtures/embeddings_demo (리포 루트 기준)


## 2. 검색 대상 - 합성 보안 공지 10건

RAG에서 "무엇을 검색하는가"는 임베딩 이전의 결정이다. 여기서는 단계 추적이 목적이므로
문서 단위를 **공지 1건 = 청크 1건**으로 두고 청킹을 생략한다. 청킹의 영향은 M3에서
따로 실험한다.

각 문서는 2~4문장짜리 한국어 요약이고, `doc_id`, 참조 `cve`, `product`, `text`를 갖는다.
**다시 강조: 아래 본문은 전부 이 노트북용으로 직접 지어낸 합성 텍스트다.** 실제 권고문
본문이 아니며, 취약점 대응의 근거로 삼으면 안 된다. CVE-ID는 문서끼리 구분되는 실제
식별자를 쓰기 위해서만 참조했다.

질의는 그중 한 건(`DOC-001`)에 대응하는 자연어 한국어 질문 1개다. 정답을 알고 있는
질의여야 회수 결과가 맞는지 눈으로 판정할 수 있다.

In [2]:
# 주의: 아래 text는 전부 합성(자작) 문장이다. 실제 공지 본문이 아니다.
DOCS = [
    {
        "doc_id": "DOC-001",
        "cve": "CVE-2021-44228",
        "product": "Apache Log4j 2",
        "text": (
            "자바 로깅 라이브러리 Log4j 2의 메시지 치환 기능에서 원격 코드 실행 취약점이 "
            "확인됐다. 공격자가 로그로 남는 입력값에 JNDI 조회 문자열을 넣으면 서버가 "
            "외부 LDAP 주소에 접속해 임의 클래스를 내려받아 실행한다. 2.15.0 이상으로 "
            "올리고, 즉시 올리기 어려우면 JndiLookup 클래스를 제거해 임시 완화한다."
        ),
    },
    {
        "doc_id": "DOC-002",
        "cve": "CVE-2014-0160",
        "product": "OpenSSL",
        "text": (
            "OpenSSL의 TLS 하트비트 확장 처리에서 요청 길이 값을 검증하지 않아 메모리가 "
            "노출되는 취약점이 확인됐다. 공격자는 실제보다 큰 길이를 보내 서버 메모리를 "
            "한 번에 최대 64KB씩 읽어낼 수 있고, 개인키와 세션 정보가 함께 빠져나갈 수 "
            "있다. 패치 후에는 인증서와 세션 키를 반드시 재발급한다."
        ),
    },
    {
        "doc_id": "DOC-003",
        "cve": "CVE-2017-0144",
        "product": "Microsoft Windows SMBv1",
        "text": (
            "윈도우 SMBv1 서버가 특수 제작된 패킷을 처리하는 과정에서 원격 코드 실행이 "
            "가능한 취약점이 확인됐다. 인증 없이 네트워크 경유로 공격할 수 있어 내부망 "
            "확산에 쓰였다. 보안 업데이트 적용과 함께 445/TCP 외부 차단, SMBv1 비활성화를 "
            "권고한다."
        ),
    },
    {
        "doc_id": "DOC-004",
        "cve": "CVE-2019-0708",
        "product": "Microsoft Windows RDP",
        "text": (
            "원격 데스크톱 서비스의 가상 채널 처리에서 인증 전 원격 코드 실행 취약점이 "
            "확인됐다. 사용자 조작 없이 악성 요청만으로 시스템 권한 실행이 가능해 웜 형태 "
            "확산 위험이 크다. 보안 업데이트를 적용하고, 지연 시 네트워크 수준 인증(NLA) "
            "활성화로 완화한다."
        ),
    },
    {
        "doc_id": "DOC-005",
        "cve": "CVE-2018-13379",
        "product": "Fortinet FortiOS SSL VPN",
        "text": (
            "FortiOS SSL VPN 웹 포털의 경로 처리에서 인증 없이 시스템 파일을 읽을 수 있는 "
            "경로 조작 취약점이 확인됐다. 평문 세션 파일이 노출되면 VPN 계정 자격증명이 "
            "그대로 유출된다. 펌웨어 업데이트 후 모든 VPN 계정 비밀번호를 재설정하고 "
            "다중 인증을 적용한다."
        ),
    },
    {
        "doc_id": "DOC-006",
        "cve": "CVE-2020-1472",
        "product": "Microsoft Windows Netlogon",
        "text": (
            "Netlogon 보안 채널의 암호화 초기화 벡터 처리 결함으로 도메인 컨트롤러 계정 "
            "비밀번호를 임의로 변경할 수 있는 권한 상승 취약점이 확인됐다. 내부망에 발판을 "
            "확보한 공격자가 도메인 관리자 권한을 획득할 수 있다. 보안 업데이트 적용 후 "
            "강제 보안 채널 모드를 단계적으로 활성화한다."
        ),
    },
    {
        "doc_id": "DOC-007",
        "cve": "CVE-2021-26855",
        "product": "Microsoft Exchange Server",
        "text": (
            "Exchange 서버의 클라이언트 액세스 서비스에서 인증 전 서버측 요청 위조(SSRF) "
            "취약점이 확인됐다. 다른 취약점과 연계하면 인증 우회 후 메일함 접근과 웹셸 "
            "설치로 이어진다. 누적 업데이트를 적용하고 침해 여부를 웹셸 탐지 스크립트로 "
            "점검한다."
        ),
    },
    {
        "doc_id": "DOC-008",
        "cve": "CVE-2022-22965",
        "product": "Spring Framework",
        "text": (
            "Spring MVC 데이터 바인딩 과정에서 클래스 로더 속성에 접근할 수 있어 원격 코드 "
            "실행으로 이어지는 취약점이 확인됐다. 특정 JDK 버전과 WAR 배포 조합에서 공격이 "
            "성립한다. 5.3.18 또는 5.2.20 이상으로 올리고, 불가 시 바인딩 금지 필드 목록을 "
            "설정한다."
        ),
    },
    {
        "doc_id": "DOC-009",
        "cve": "CVE-2023-4863",
        "product": "libwebp",
        "text": (
            "WebP 이미지 디코딩 라이브러리 libwebp의 허프만 테이블 처리에서 힙 버퍼 오버플로 "
            "취약점이 확인됐다. 악성 이미지를 열기만 해도 코드 실행이 가능해 브라우저와 "
            "메신저 등 라이브러리를 내장한 제품 전반이 영향을 받는다. 각 제품 벤더가 배포한 "
            "업데이트를 적용한다."
        ),
    },
    {
        "doc_id": "DOC-010",
        "cve": "CVE-2024-3400",
        "product": "Palo Alto Networks PAN-OS",
        "text": (
            "PAN-OS GlobalProtect 게이트웨이 기능에서 인증 없이 명령을 주입할 수 있는 "
            "취약점이 확인됐다. 공격자가 root 권한으로 임의 명령을 실행해 장비를 장악할 수 "
            "있다. 핫픽스 적용과 함께 장비 로그·설정 변경 이력을 점검하고 침해 시 자격증명을 "
            "전량 교체한다."
        ),
    },
]

QUERY = "자바 로깅 라이브러리에서 JNDI 조회로 원격 코드가 실행되는 취약점은 어떻게 대응하나요?"
EXPECTED_DOC_ID = "DOC-001"  # 사람이 미리 정한 정답 (약한 라벨이 아니라 이 노트북의 설계값)

print(f"문서 {len(DOCS)}건 / 질의 1건")
print(f"질의   : {QUERY}")
print(f"기대 정답: {EXPECTED_DOC_ID}\n")
for d in DOCS:
    print(f"{d['doc_id']}  {d['cve']:<16} {len(d['text']):>3}자  {d['product']}")

문서 10건 / 질의 1건
질의   : 자바 로깅 라이브러리에서 JNDI 조회로 원격 코드가 실행되는 취약점은 어떻게 대응하나요?
기대 정답: DOC-001

DOC-001  CVE-2021-44228   183자  Apache Log4j 2
DOC-002  CVE-2014-0160    169자  OpenSSL
DOC-003  CVE-2017-0144    144자  Microsoft Windows SMBv1
DOC-004  CVE-2019-0708    145자  Microsoft Windows RDP
DOC-005  CVE-2018-13379   152자  Fortinet FortiOS SSL VPN
DOC-006  CVE-2020-1472    161자  Microsoft Windows Netlogon
DOC-007  CVE-2021-26855   141자  Microsoft Exchange Server
DOC-008  CVE-2022-22965   155자  Spring Framework
DOC-009  CVE-2023-4863    151자  libwebp
DOC-010  CVE-2024-3400    153자  Palo Alto Networks PAN-OS


## 3. 모델 적재 - 무엇이 들어 있는지부터 본다

`SentenceTransformer`는 모듈 리스트다. `print(model)`이 찍는 세 줄이 곧 이 노트북에서
손으로 재현할 파이프라인이다.

1. `Transformer` - 토큰화 + 인코더 forward
2. `Pooling` - 토큰 벡터 묶음을 문장 벡터 1개로 축약 (bge-m3 dense는 `cls`)
3. `Normalize` - L2 정규화

bge-m3는 dense/sparse/multi-vector 세 가지 표현을 내는 모델이지만,
`SentenceTransformer`로 불러오면 이 셋 중 **dense만** 쓰는 구성이 된다. 이 노트북과
이어지는 #6은 dense only 기준선을 다룬다.

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_NAME, revision=MODEL_REVISION, device=DEVICE)
model.eval()

tokenizer = model.tokenizer
encoder = model[0].auto_model
pooling_cfg = model[1].get_config_dict()

print(model)
print()
print("model name     :", MODEL_NAME)
print("revision       :", MODEL_REVISION)
print("max_seq_length :", model.max_seq_length)
print("pooling        :", {k: v for k, v in pooling_cfg.items() if v is True or "dim" in k})
print("hidden_size    :", encoder.config.hidden_size)
print("vocab_size     :", encoder.config.vocab_size)
print("tokenizer      :", type(tokenizer).__name__)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 59106.64it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

model name     : BAAI/bge-m3
revision       : 5617a9f61b028005a4858fdac845db406aefb181
max_seq_length : 8192
pooling        : {'embedding_dimension': 1024, 'include_prompt': True}
hidden_size    : 1024
vocab_size     : 250002
tokenizer      : XLMRobertaTokenizer


## 4. 단계 1 - 토큰화

모델은 문자열을 보지 않는다. 토크나이저가 문자열을 정수 ID 배열로 바꾸고, 그 ID가
임베딩 행렬의 행 번호로 쓰인다. 이 단계에서 확인할 것은 세 가지다.

- **어떤 단위로 쪼개지는가.** bge-m3는 XLM-RoBERTa 계열 SentencePiece 토크나이저를 쓴다.
  한국어는 대체로 형태소보다 잘게 쪼개지고, 토큰 앞의 `▁`가 공백(어절 시작)을 뜻한다.
- **특수 토큰.** 앞의 `<s>`(id 0)가 CLS 자리, 뒤의 `</s>`(id 2)가 SEP다. pooling에서 쓸
  벡터가 바로 이 첫 번째 자리의 출력이므로 위치를 눈으로 확인해 둔다.
- **길이.** 토큰 수가 곧 비용이고, `max_seq_length`를 넘으면 뒤가 잘린다.

In [4]:
doc = DOCS[0]

enc_query = tokenizer(QUERY, return_tensors="pt")
enc_doc = tokenizer(doc["text"], return_tensors="pt")

for label, text, enc in [("질의", QUERY, enc_query), (doc["doc_id"], doc["text"], enc_doc)]:
    ids = enc["input_ids"][0]
    toks = tokenizer.convert_ids_to_tokens(ids)
    print(f"[{label}] 원문 {len(text)}자 -> 토큰 {len(ids)}개")
    print("  input_ids[:12] :", ids[:12].tolist())
    print("  tokens[:12]    :", toks[:12])
    print("  tokens[-4:]    :", toks[-4:])
    print("  attention_mask :", enc["attention_mask"][0][:12].tolist(), "...")
    print()

print("특수 토큰 id:",
      {"cls/bos": tokenizer.cls_token_id,
       "sep/eos": tokenizer.sep_token_id,
       "pad": tokenizer.pad_token_id,
       "unk": tokenizer.unk_token_id})
print(f"질의 1자당 토큰 수: {len(enc_query['input_ids'][0]) / len(QUERY):.2f}")
print(f"{doc['doc_id']} 1자당 토큰 수: {len(enc_doc['input_ids'][0]) / len(doc['text']):.2f}")

[질의] 원문 50자 -> 토큰 30개
  input_ids[:12] : [0, 8511, 12775, 29770, 243902, 6, 105203, 7591, 2085, 1180, 6, 135513]
  tokens[:12]    : ['<s>', '▁자', '바', '▁로', '깅', '▁', '라이브', '러', '리', '에서', '▁', 'JN']
  tokens[-4:]    : ['하나', '요', '?', '</s>']
  attention_mask : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1] ...

[DOC-001] 원문 183자 -> 토큰 101개
  input_ids[:12] : [0, 8511, 12775, 29770, 243902, 6, 105203, 7591, 2085, 13146, 617, 170]
  tokens[:12]    : ['<s>', '▁자', '바', '▁로', '깅', '▁', '라이브', '러', '리', '▁Log', '4', 'j']
  tokens[-4:]    : ['화', '한다', '.', '</s>']
  attention_mask : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1] ...

특수 토큰 id: {'cls/bos': 0, 'sep/eos': 2, 'pad': 1, 'unk': 3}
질의 1자당 토큰 수: 0.60
DOC-001 1자당 토큰 수: 0.55


### 4-1. 길이 제한(truncation)은 조용히 일어난다

`model.max_seq_length`는 8192다. bge-m3의 위치 임베딩이 8192까지를 학습했기 때문이고,
긴 문서를 통째로 넣을 수 있다는 것이 이 모델의 강점 중 하나다. 다만 **한도를 넘으면
예외가 아니라 절단**이다. 뒤쪽 내용이 사라진 채 벡터가 만들어지고, 검색 결과만 이상해진다.

아래는 같은 텍스트를 한도 32로 잘라 보며 절단이 어떻게 일어나는지를 눈으로 확인하는
실험이다. 잘린 뒤에도 마지막 자리는 `</s>`로 닫힌다는 점에 주의한다 - 겉보기에는 멀쩡한
입력이라 로그만 봐서는 알아채기 어렵다.

In [5]:
long_text = doc["text"] * 40  # 한도를 확실히 넘기려고 같은 문단을 반복
full = tokenizer(long_text, return_tensors="pt")
capped = tokenizer(long_text, truncation=True, max_length=model.max_seq_length,
                   return_tensors="pt")
tiny = tokenizer(long_text, truncation=True, max_length=32, return_tensors="pt")

print("truncation 없이        :", full["input_ids"].shape[1], "토큰")
print("max_length=8192 적용   :", capped["input_ids"].shape[1], "토큰",
      "(절단 발생)" if capped["input_ids"].shape[1] < full["input_ids"].shape[1] else "(절단 없음)")
print("max_length=32 적용     :", tiny["input_ids"].shape[1], "토큰")
print("  잘린 뒤 tokens[-4:] :", tokenizer.convert_ids_to_tokens(tiny["input_ids"][0])[-4:])
print()
print("이 노트북의 문서 10건 토큰 길이:")
lengths = [len(tokenizer(d["text"])["input_ids"]) for d in DOCS]
for d, n in zip(DOCS, lengths, strict=True):
    print(f"  {d['doc_id']}: {n:>3} 토큰  (한도 {model.max_seq_length} 대비 {n / 8192:.2%})")
print(f"최대 {max(lengths)} 토큰 - 전부 한도 안이라 이 노트북에서는 절단이 일어나지 않는다.")

truncation 없이        : 3962 토큰
max_length=8192 적용   : 3962 토큰 (절단 없음)
max_length=32 적용     : 32 토큰
  잘린 뒤 tokens[-4:] : ['.', '▁공격', '자가', '</s>']

이 노트북의 문서 10건 토큰 길이:
  DOC-001: 101 토큰  (한도 8192 대비 1.23%)
  DOC-002:  89 토큰  (한도 8192 대비 1.09%)
  DOC-003:  72 토큰  (한도 8192 대비 0.88%)
  DOC-004:  71 토큰  (한도 8192 대비 0.87%)
  DOC-005:  72 토큰  (한도 8192 대비 0.88%)
  DOC-006:  86 토큰  (한도 8192 대비 1.05%)
  DOC-007:  77 토큰  (한도 8192 대비 0.94%)
  DOC-008:  75 토큰  (한도 8192 대비 0.92%)
  DOC-009:  84 토큰  (한도 8192 대비 1.03%)
  DOC-010:  77 토큰  (한도 8192 대비 0.94%)
최대 101 토큰 - 전부 한도 안이라 이 노트북에서는 절단이 일어나지 않는다.


## 5. 단계 2 - 인코더 forward

토큰 ID가 인코더를 통과하면 **토큰마다 하나씩** 1024차원 벡터가 나온다.
`last_hidden_state`의 shape는 `(batch, seq_len, hidden)`이다. 여기가 핵심이다:
이 시점에는 아직 "문장 벡터"가 없다. 문장당 벡터 1개는 다음 단계인 pooling이 만든다.

In [6]:
with torch.no_grad():
    out_query = encoder(**enc_query)
    out_doc = encoder(**enc_doc)

hs_query = out_query.last_hidden_state
hs_doc = out_doc.last_hidden_state

print("질의 last_hidden_state :", tuple(hs_query.shape), hs_query.dtype)
print("문서 last_hidden_state :", tuple(hs_doc.shape), hs_doc.dtype)
print()
print("shape 해석: (batch=1, seq_len=토큰 수, hidden=1024)")
print("질의 0번 토큰(CLS 자리) 벡터 앞 8개:", hs_query[0, 0, :8].tolist())
print("질의 1번 토큰 벡터 앞 8개         :", hs_query[0, 1, :8].tolist())
print()
print("이 단계에서 문장 벡터는 아직 없다. 토큰 벡터가",
      hs_query.shape[1], "개 있을 뿐이다.")

질의 last_hidden_state : (1, 30, 1024) torch.float32
문서 last_hidden_state : (1, 101, 1024) torch.float32

shape 해석: (batch=1, seq_len=토큰 수, hidden=1024)
질의 0번 토큰(CLS 자리) 벡터 앞 8개: [-0.635053277015686, -0.3220037817955017, -1.0709915161132812, 1.0753819942474365, -0.736811101436615, 0.8059067726135254, 0.9799438714981079, -0.08960150927305222]
질의 1번 토큰 벡터 앞 8개         : [0.453488826751709, -0.43408915400505066, -0.7155040502548218, 0.8827723860740662, -0.7809284925460815, 0.9991286993026733, 1.0452818870544434, -0.16308903694152832]

이 단계에서 문장 벡터는 아직 없다. 토큰 벡터가 30 개 있을 뿐이다.


## 6. 단계 3 - pooling: 토큰 N개를 벡터 1개로

토큰 벡터 묶음을 문장 벡터 1개로 줄이는 방법은 여러 가지고, **모델마다 학습 때 쓴 방식이
정해져 있다.** bge-m3 dense는 `cls` pooling이다(위 3절에서 확인한 `pooling_mode: cls`).
0번 자리, 즉 `<s>` 토큰의 출력 벡터를 그대로 문장 벡터로 쓴다.

왜 이걸 따져야 하나. 평균(mean) pooling도 문장 벡터를 만들어 내고, 코드는 돌아가고,
검색도 되는 것처럼 보인다. 하지만 학습 때 쓴 pooling과 다르면 그 벡터는 모델이 정렬해 둔
공간과 어긋난다. 아래에서 두 방식의 코사인 유사도를 실제 수치로 확인한다.

In [7]:
def cls_pool(hidden: torch.Tensor) -> torch.Tensor:
    """bge-m3 dense가 쓰는 방식: 0번 토큰(CLS) 벡터를 그대로 문장 벡터로 삼는다."""
    return hidden[:, 0]


def mean_pool(hidden: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """대조군: 패딩을 제외한 토큰 벡터의 산술 평균."""
    mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
    return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)


def cosine(a: torch.Tensor, b: torch.Tensor) -> float:
    return float(torch.nn.functional.cosine_similarity(a, b, dim=-1).item())


q_cls = cls_pool(hs_query)
q_mean = mean_pool(hs_query, enc_query["attention_mask"])
d_cls = cls_pool(hs_doc)
d_mean = mean_pool(hs_doc, enc_doc["attention_mask"])

print("질의 CLS  벡터 shape:", tuple(q_cls.shape), "앞 5개:", q_cls[0, :5].tolist())
print("질의 mean 벡터 shape:", tuple(q_mean.shape), "앞 5개:", q_mean[0, :5].tolist())
print()
print(f"같은 질의의 CLS vs mean 코사인 : {cosine(q_cls, q_mean):+.4f}")
print(f"같은 문서의 CLS vs mean 코사인 : {cosine(d_cls, d_mean):+.4f}")
print()
print(f"질의-문서 유사도 (CLS  경로)  : {cosine(q_cls, d_cls):+.4f}  <- 모델이 학습한 방식")
print(f"질의-문서 유사도 (mean 경로)  : {cosine(q_mean, d_mean):+.4f}  <- 학습과 다른 방식")
print()
print("같은 입력, 같은 forward인데도 pooling만 바꾸면 벡터가 다른 방향을 가리킨다.")
print("pooling은 '구현 취향'이 아니라 모델 사양의 일부다.")

질의 CLS  벡터 shape: (1, 1024) 앞 5개: [-0.635053277015686, -0.3220037817955017, -1.0709915161132812, 1.0753819942474365, -0.736811101436615]
질의 mean 벡터 shape: (1, 1024) 앞 5개: [-0.49174049496650696, -0.43981412053108215, -0.5032684803009033, 0.5583924055099487, -0.3668018877506256]

같은 질의의 CLS vs mean 코사인 : +0.7695
같은 문서의 CLS vs mean 코사인 : +0.7270

질의-문서 유사도 (CLS  경로)  : +0.7857  <- 모델이 학습한 방식
질의-문서 유사도 (mean 경로)  : +0.8806  <- 학습과 다른 방식

같은 입력, 같은 forward인데도 pooling만 바꾸면 벡터가 다른 방향을 가리킨다.
pooling은 '구현 취향'이 아니라 모델 사양의 일부다.


### 6-1. 위 수치를 어떻게 읽어야 하나

mean 경로의 질의-문서 유사도가 CLS 경로보다 **높게** 나오는데, 이걸 "mean이 더 좋다"로
읽으면 틀린다. 유사도 절댓값은 품질 지표가 아니다. 토큰 벡터를 평균 내면 모든 문장이
말뭉치 평균 방향으로 끌려가서(anisotropy) 관련 없는 문장끼리도 점수가 같이 올라간다.
검색에서 중요한 것은 정답과 오답의 **간격**이지 정답 점수의 절댓값이 아니다.

그래서 pooling 방식을 바꿀 때 확인해야 할 것은 "점수가 올랐나"가 아니라
"정답이 여전히 1위이고 2위와의 차이가 벌어졌나"다. 아래 10절에서 CLS 경로의 1-2위
점수 차(margin)를 함께 출력하는 이유가 그것이다. 이 노트북은 기준선 정의가 목적이므로
모델이 학습한 방식(CLS)을 그대로 따르고, 방식 교체의 효과 비교는 평가셋이 생기는
M2 이후에 한다.

## 7. 단계 4 - L2 정규화

pooling 결과는 길이(norm)가 제각각이다. 벡터를 자기 L2 norm으로 나누면 모두 단위
구면 위에 놓이고, 그때부터 **내적이 곧 코사인 유사도**가 된다. 이것이 실무에서 중요한
이유는 두 가지다.

- 유사도 계산이 행렬 곱 한 번으로 끝난다(나눗셈이 사라진다).
- Qdrant 같은 벡터 DB에서 `Cosine`과 `Dot` 거리 설정이 같은 결과를 내게 된다. 다만
  정규화를 안 한 벡터를 `Dot`으로 넣으면 긴 벡터가 무조건 유리해진다. #6에서 이 경계를
  다시 확인한다.

`SentenceTransformer`는 `Normalize` 모듈로 이 단계를 자동 수행한다. 아래에서는
그 자동 단계를 손으로 재현해 전/후 값을 비교한다.

In [8]:
def l2_normalize(vec: torch.Tensor) -> torch.Tensor:
    return vec / vec.norm(p=2, dim=-1, keepdim=True)


q_norm = l2_normalize(q_cls)
d_norm = l2_normalize(d_cls)

print(f"정규화 전 질의 norm : {q_cls.norm().item():.6f}")
print(f"정규화 전 문서 norm : {d_cls.norm().item():.6f}")
print(f"정규화 후 질의 norm : {q_norm.norm().item():.6f}")
print(f"정규화 후 문서 norm : {d_norm.norm().item():.6f}")
print()
print("정규화 전 질의 벡터 앞 5개:", [round(x, 5) for x in q_cls[0, :5].tolist()])
print("정규화 후 질의 벡터 앞 5개:", [round(x, 5) for x in q_norm[0, :5].tolist()])
print()
dot_raw = float((q_cls[0] @ d_cls[0]).item())
dot_norm = float((q_norm[0] @ d_norm[0]).item())
print(f"정규화 전 내적 : {dot_raw:+.6f}  (코사인이 아니다 - 벡터 길이가 섞여 있다)")
print(f"정규화 후 내적 : {dot_norm:+.6f}  (= 코사인 유사도)")
print(f"직접 계산한 코사인: {cosine(q_cls, d_cls):+.6f}  -> 정규화 후 내적과 같다")

정규화 전 질의 norm : 26.122885
정규화 전 문서 norm : 26.133236
정규화 후 질의 norm : 1.000000
정규화 후 문서 norm : 1.000000

정규화 전 질의 벡터 앞 5개: [-0.63505, -0.322, -1.07099, 1.07538, -0.73681]
정규화 후 질의 벡터 앞 5개: [-0.02431, -0.01233, -0.041, 0.04117, -0.02821]

정규화 전 내적 : +536.367249  (코사인이 아니다 - 벡터 길이가 섞여 있다)
정규화 후 내적 : +0.785684  (= 코사인 유사도)
직접 계산한 코사인: +0.785684  -> 정규화 후 내적과 같다


## 8. 단계 5 - 최종 벡터의 신원

벡터를 저장소에 넣기 전에 매번 확인해야 하는 세 가지: **차원, dtype, norm**.
차원이 다르면 컬렉션이 거부하고, dtype이 다르면 저장 용량과 정밀도가 달라지며,
norm이 1이 아니면 코사인/내적 가정이 깨진다.

In [9]:
query_vec = q_norm[0].numpy().astype(np.float32)
doc_vec = d_norm[0].numpy().astype(np.float32)

for name, v in [("질의", query_vec), (doc["doc_id"], doc_vec)]:
    print(f"[{name}] shape={v.shape} dtype={v.dtype} "
          f"norm={np.linalg.norm(v):.6f} "
          f"min={v.min():+.4f} max={v.max():+.4f} bytes={v.nbytes}")
print()
print("차원 1024는 모델 config의 hidden_size와 같다:", encoder.config.hidden_size)
print("float32 1024차원 = 4KB/벡터. 문서 10만 건이면 약 390MB.")

[질의] shape=(1024,) dtype=float32 norm=1.000000 min=-0.1508 max=+0.2212 bytes=4096
[DOC-001] shape=(1024,) dtype=float32 norm=1.000000 min=-0.1517 max=+0.2101 bytes=4096

차원 1024는 모델 config의 hidden_size와 같다: 1024
float32 1024차원 = 4KB/벡터. 문서 10만 건이면 약 390MB.


## 9. 교차검증 - 손으로 간 길과 `model.encode()`가 같은 곳에 도착하는가

여기까지가 이 노트북의 핵심 주장이다: `model.encode()` 한 줄은 지금까지 밟은
**토큰화 → forward → CLS pooling → L2 정규화**와 정확히 같은 일을 한다. 주장으로
끝내지 않고 `assert`로 증명한다. 이 assert가 깨지면 위 설명 어딘가가 틀린 것이므로,
노트북이 실행되는 한 설명과 코드가 어긋날 수 없다.

`atol=1e-5`는 부동소수점 연산 순서 차이를 허용하는 값이다. 두 경로는 수학적으로
같지만 연산이 묶이는 순서가 달라 마지막 자리가 흔들릴 수 있다.

In [10]:
manual_query = query_vec
manual_doc = doc_vec

auto_query = model.encode([QUERY], normalize_embeddings=True,
                          convert_to_numpy=True).astype(np.float32)[0]
auto_doc = model.encode([doc["text"]], normalize_embeddings=True,
                        convert_to_numpy=True).astype(np.float32)[0]

for name, manual, auto in [("질의", manual_query, auto_query),
                           (doc["doc_id"], manual_doc, auto_doc)]:
    diff = np.abs(manual - auto)
    cos = float(manual @ auto)
    print(f"[{name}] 최대 절대오차={diff.max():.3e} 평균={diff.mean():.3e} 코사인={cos:.8f}")
    assert np.allclose(manual, auto, atol=1e-5), f"{name}: 수동 경로와 encode() 결과가 다르다"

print()
print("assert 통과: 수동 경로(tokenizer -> forward -> CLS -> L2)와 model.encode()가 동일하다.")

[질의] 최대 절대오차=2.980e-08 평균=2.739e-09 코사인=1.00000000
[DOC-001] 최대 절대오차=1.490e-08 평균=2.059e-09 코사인=0.99999988

assert 통과: 수동 경로(tokenizer -> forward -> CLS -> L2)와 model.encode()가 동일하다.


## 10. 단계 6 - NumPy 전수 코사인

10건짜리 코퍼스에서는 근사 탐색(ANN)이 필요 없다. 전수(brute-force) 계산이
**정답 그 자체**이고, #6에서 Qdrant의 exact 검색과 HNSW 근사 검색을 이 값과 대조하게
된다. ANN이 얼마나 정확한지를 말하려면 먼저 기준이 있어야 한다.

계산은 단순하다. 문서 벡터를 쌓아 `D`(10x1024), 질의 벡터를 `q`(1024)라 하면

```
score[i] = sum over j of D[i][j] * q[j]      # 1024개 곱의 합
scores   = D @ q                              # 위를 10행에 대해 한 번에
```

모든 벡터가 단위 길이이므로 이 내적이 곧 코사인 유사도다. 아래에서 행렬 곱 결과와
파이썬 루프로 한 항씩 더한 결과가 같은지 확인한다.

In [11]:
doc_texts = [d["text"] for d in DOCS]
doc_matrix = model.encode(doc_texts, normalize_embeddings=True, convert_to_numpy=True,
                          batch_size=4, show_progress_bar=False).astype(np.float32)
query_matrix = model.encode([QUERY], normalize_embeddings=True,
                            convert_to_numpy=True).astype(np.float32)
q = query_matrix[0]

print("문서 행렬 D :", doc_matrix.shape, doc_matrix.dtype)
print("질의 벡터 q :", q.shape, q.dtype)
print("각 행의 norm:", np.round(np.linalg.norm(doc_matrix, axis=1), 6).tolist())
print()

scores = doc_matrix @ q  # (10, 1024) @ (1024,) -> (10,)

# 행렬 곱이 실제로 무슨 계산인지 첫 문서에 대해 손으로 재현한다.
manual_score = 0.0
for j in range(doc_matrix.shape[1]):
    manual_score += float(doc_matrix[0, j]) * float(q[j])
print(f"DOC-001 내적: 1024개 곱을 더한 값 = {manual_score:+.6f}")
print(f"DOC-001 내적: D @ q 결과          = {float(scores[0]):+.6f}")
assert np.isclose(manual_score, float(scores[0]), atol=1e-5), "루프와 행렬 곱이 다르다"

# 정규화를 안 했다면 나눗셈이 필요하다는 것도 같이 확인한다.
cos_explicit = (doc_matrix @ q) / (np.linalg.norm(doc_matrix, axis=1) * np.linalg.norm(q))
assert np.allclose(scores, cos_explicit, atol=1e-6)
print("정규화 덕분에 내적 = 코사인 (나눗셈 생략 가능) 확인")
print()
print("전체 점수:")
for d, s in sorted(zip(DOCS, scores, strict=True), key=lambda p: -p[1]):
    print(f"  {float(s):+.6f}  {d['doc_id']}  {d['cve']:<16} {d['product']}")

문서 행렬 D : (10, 1024) float32
질의 벡터 q : (1024,) float32
각 행의 norm: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

DOC-001 내적: 1024개 곱을 더한 값 = +0.785684
DOC-001 내적: D @ q 결과          = +0.785684
정규화 덕분에 내적 = 코사인 (나눗셈 생략 가능) 확인

전체 점수:
  +0.785684  DOC-001  CVE-2021-44228   Apache Log4j 2
  +0.613373  DOC-008  CVE-2022-22965   Spring Framework
  +0.602629  DOC-004  CVE-2019-0708    Microsoft Windows RDP
  +0.542555  DOC-003  CVE-2017-0144    Microsoft Windows SMBv1
  +0.494671  DOC-007  CVE-2021-26855   Microsoft Exchange Server
  +0.492800  DOC-005  CVE-2018-13379   Fortinet FortiOS SSL VPN
  +0.473131  DOC-002  CVE-2014-0160    OpenSSL
  +0.464280  DOC-009  CVE-2023-4863    libwebp
  +0.457798  DOC-006  CVE-2020-1472    Microsoft Windows Netlogon
  +0.448973  DOC-010  CVE-2024-3400    Palo Alto Networks PAN-OS


## 11. 단계 7 - top-3 회수와 원문 확인

점수 벡터에서 상위 k개를 뽑는다. `np.argsort`는 오름차순이므로 음수를 붙여 내림차순으로
만든다(값이 아니라 **인덱스**가 나온다는 점이 핵심 - 그 인덱스로 원문을 되찾는다).

RAG에서 이 단계의 산출물은 점수가 아니라 **LLM에 넘길 원문 조각**이다. 그래서 점수와
문서 ID만 찍고 끝내면 안 되고, 실제로 어떤 문장이 회수됐는지를 같이 봐야 한다.

In [12]:
TOP_K = 3
order = np.argsort(-scores)[:TOP_K]

print(f"질의: {QUERY}\n")
print(f"top-{TOP_K} 회수 결과")
print("=" * 78)
for rank, idx in enumerate(order, start=1):
    d = DOCS[int(idx)]
    print(f"[{rank}위] score={float(scores[idx]):+.6f}  "
          f"corpus_index={int(idx)}  doc_id={d['doc_id']}  cve={d['cve']}")
    print(f"       product: {d['product']}")
    print(f"       원문(합성): {d['text'][:90]}...")
    print("-" * 78)

top1 = DOCS[int(order[0])]
margin = float(scores[order[0]] - scores[order[1]])
print(f"1위 문서 = {top1['doc_id']}, 기대 정답 = {EXPECTED_DOC_ID}")
print(f"1-2위 점수 차(margin) = {margin:+.6f}")
assert top1["doc_id"] == EXPECTED_DOC_ID, "기대한 문서가 1위로 회수되지 않았다"
print("hit@1 통과")

질의: 자바 로깅 라이브러리에서 JNDI 조회로 원격 코드가 실행되는 취약점은 어떻게 대응하나요?

top-3 회수 결과
[1위] score=+0.785684  corpus_index=0  doc_id=DOC-001  cve=CVE-2021-44228
       product: Apache Log4j 2
       원문(합성): 자바 로깅 라이브러리 Log4j 2의 메시지 치환 기능에서 원격 코드 실행 취약점이 확인됐다. 공격자가 로그로 남는 입력값에 JNDI 조회 문자열을 넣으면 서버가...
------------------------------------------------------------------------------
[2위] score=+0.613373  corpus_index=7  doc_id=DOC-008  cve=CVE-2022-22965
       product: Spring Framework
       원문(합성): Spring MVC 데이터 바인딩 과정에서 클래스 로더 속성에 접근할 수 있어 원격 코드 실행으로 이어지는 취약점이 확인됐다. 특정 JDK 버전과 WAR 배포 조...
------------------------------------------------------------------------------
[3위] score=+0.602629  corpus_index=3  doc_id=DOC-004  cve=CVE-2019-0708
       product: Microsoft Windows RDP
       원문(합성): 원격 데스크톱 서비스의 가상 채널 처리에서 인증 전 원격 코드 실행 취약점이 확인됐다. 사용자 조작 없이 악성 요청만으로 시스템 권한 실행이 가능해 웜 형태 확산...
------------------------------------------------------------------------------
1위 문서 = DOC-001, 기대 정답 = DOC-001
1-2위 점수 차(margi

## 12. #6으로 넘기는 fixture

이슈 [#6](https://github.com/Bkankim/krsec-rag-lab/issues/6)은 **같은 문서·같은 질의**로
Qdrant에 적재해 exact 검색과 HNSW 근사 검색 결과를 이 노트북의 NumPy 전수 점수와
대조한다. 대조가 성립하려면 #6이 임베딩을 다시 만들면 안 된다 - 같은 벡터를 읽어 써야
"검색 결과 차이"가 검색 방식의 차이로만 설명된다.

저장 위치: `fixtures/embeddings_demo/`

| 파일 | 내용 |
|---|---|
| `documents.json` | 합성 문서 10건 + 질의 + 기대 정답 doc_id |
| `doc_embeddings.npy` | 문서 임베딩 (10, 1024) float32, L2 정규화됨 |
| `query_embedding.npy` | 질의 임베딩 (1024,) float32, L2 정규화됨 |
| `baseline_scores.json` | NumPy 전수 코사인 점수와 top-3 순위 |
| `manifest.json` | 모델명·revision·정규화 여부·생성 시각·본문 해시 |

manifest는 PLAN.md 원칙 5(재현성)를 이 산출물 단위에서 지키기 위한 것이다. 벡터 파일만
남으면 반년 뒤에 "이게 어느 모델의 출력인지" 알 길이 없다.

In [13]:
FIXTURE_DIR.mkdir(parents=True, exist_ok=True)

corpus_payload = {
    "note": (
        "아래 text는 이 리포용으로 직접 작성한 합성 한국어 보안 공지 요약이다. "
        "실제 KISA/NVD 권고문 본문이 아니며 사실 확인 용도로 쓸 수 없다. "
        "cve/product 필드만 실제 공개 식별자를 참조한다."
    ),
    "query": QUERY,
    "expected_doc_id": EXPECTED_DOC_ID,
    "documents": DOCS,
}
corpus_bytes = json.dumps(corpus_payload, ensure_ascii=False, indent=2,
                          sort_keys=True).encode("utf-8")

manifest = {
    "created_at": datetime.now(UTC).isoformat(timespec="seconds"),
    "source_notebook": "notebooks/00_embeddings_from_scratch.ipynb",
    "issue": 5,
    "model": {
        "name": MODEL_NAME,
        "revision": MODEL_REVISION,
        "pooling": "cls",
        "normalized": True,
        "dimension": int(doc_matrix.shape[1]),
        "dtype": str(doc_matrix.dtype),
        "max_seq_length": int(model.max_seq_length),
        "representation": "dense only (bge-m3의 sparse/colbert 헤드는 사용하지 않음)",
    },
    "runtime": {
        "device": DEVICE,
        "seed": SEED,
        "torch": torch.__version__,
        "numpy": np.__version__,
        "python": platform.python_version(),
    },
    "corpus": {
        "n_documents": len(DOCS),
        "synthetic": True,
        "documents_sha256": hashlib.sha256(corpus_bytes).hexdigest(),
    },
}

baseline = {
    "metric": "cosine (= dot product on L2-normalized vectors)",
    "method": "numpy brute-force full scan",
    "scores": {d["doc_id"]: round(float(s), 8) for d, s in zip(DOCS, scores, strict=True)},
    "top_k": [
        {"rank": r, "doc_id": DOCS[int(i)]["doc_id"], "score": round(float(scores[i]), 8)}
        for r, i in enumerate(np.argsort(-scores)[:TOP_K], start=1)
    ],
}

(FIXTURE_DIR / "documents.json").write_bytes(corpus_bytes)
(FIXTURE_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(FIXTURE_DIR / "baseline_scores.json").write_text(
    json.dumps(baseline, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
np.save(FIXTURE_DIR / "doc_embeddings.npy", doc_matrix)
np.save(FIXTURE_DIR / "query_embedding.npy", q)

print("저장 위치:", FIXTURE_DIR.relative_to(REPO_ROOT))
for p in sorted(FIXTURE_DIR.iterdir()):
    print(f"  {p.name:<24} {p.stat().st_size:>8,} bytes")
print()

# 저장한 파일을 다시 읽어 동일성을 확인한다. 저장 단계도 검증 대상이다.
reloaded = np.load(FIXTURE_DIR / "doc_embeddings.npy")
reloaded_q = np.load(FIXTURE_DIR / "query_embedding.npy")
assert reloaded.shape == (len(DOCS), 1024) and reloaded.dtype == np.float32
assert np.array_equal(reloaded, doc_matrix) and np.array_equal(reloaded_q, q)
print("재적재 검증 통과:", reloaded.shape, reloaded.dtype)
print("documents.json sha256:", manifest["corpus"]["documents_sha256"][:16], "...")

저장 위치: fixtures/embeddings_demo
  baseline_scores.json          662 bytes
  doc_embeddings.npy         41,088 bytes
  documents.json              5,254 bytes
  manifest.json                 735 bytes
  query_embedding.npy         4,224 bytes

재적재 검증 통과: (10, 1024) float32
documents.json sha256: 244451c38d850241 ...


## 13. 한 줄 추적 요약

#6에서 이어붙일 수 있도록, 질의 1건이 지나온 경로를 한 표로 모은다.

In [14]:
trace = [
    ("0. 입력", f"한국어 질의 {len(QUERY)}자"),
    ("1. 토큰화", (f"input_ids {tuple(enc_query['input_ids'].shape)} "
                  f"({len(enc_query['input_ids'][0])} 토큰, 한도 {model.max_seq_length})")),
    ("2. forward", f"last_hidden_state {tuple(hs_query.shape)}"),
    ("3. pooling", (f"CLS(0번 토큰) 선택 -> {tuple(q_cls.shape)}, "
                    f"mean과의 코사인 {cosine(q_cls, q_mean):+.4f}")),
    ("4. 정규화", f"norm {q_cls.norm().item():.4f} -> {q_norm.norm().item():.4f}"),
    ("5. 최종 벡터", f"shape {q.shape} dtype {q.dtype} norm {np.linalg.norm(q):.6f}"),
    ("6. 교차검증", "수동 경로 vs model.encode() allclose(atol=1e-5) 통과"),
    ("7. 전수 코사인", f"D{doc_matrix.shape} @ q -> 점수 {scores.shape}"),
    ("8. top-1", (f"{top1['doc_id']} ({top1['cve']}) score {float(scores[order[0]]):+.6f}, "
                  f"margin {margin:+.6f}")),
    ("9. 인계", "fixtures/embeddings_demo/ -> 이슈 #6(Qdrant exact/HNSW 대조)"),
]
width = max(len(k) for k, _ in trace)
for k, v in trace:
    print(f"{k:<{width}}  |  {v}")

0. 입력       |  한국어 질의 50자
1. 토큰화      |  input_ids (1, 30) (30 토큰, 한도 8192)
2. forward  |  last_hidden_state (1, 30, 1024)
3. pooling  |  CLS(0번 토큰) 선택 -> (1, 1024), mean과의 코사인 +0.7695
4. 정규화      |  norm 26.1229 -> 1.0000
5. 최종 벡터    |  shape (1024,) dtype float32 norm 1.000000
6. 교차검증     |  수동 경로 vs model.encode() allclose(atol=1e-5) 통과
7. 전수 코사인   |  D(10, 1024) @ q -> 점수 (10,)
8. top-1    |  DOC-001 (CVE-2021-44228) score +0.785684, margin +0.172311
9. 인계       |  fixtures/embeddings_demo/ -> 이슈 #6(Qdrant exact/HNSW 대조)


## 14. 내재화 게이트

이 노트북의 완료 조건은 "실행됐다"가 아니라 **"노트북을 보지 않고 설명할 수 있다"**이다
(이슈 #5 완료 기준). 아래 질문에 노트북·검색 없이 본인 말로 답하고, 통과 여부를
이슈 #5 코멘트로 기록한다. 여기에 답을 적어 두면 게이트가 무력화되므로 답은 넣지 않는다.

1. 이 문서 1건이 문자열에서 1024차원 float32 벡터가 되기까지 거친 단계를 순서대로 말하고,
   각 단계에서 데이터의 shape가 어떻게 바뀌는지 설명하라.
2. `last_hidden_state`의 shape가 `(1, 87, 1024)`라고 하자. 세 숫자가 각각 무엇이고,
   이 시점에 "문장 벡터"가 아직 없다는 말은 무슨 뜻인가?
3. bge-m3 dense는 CLS pooling을 쓴다. mean pooling으로 바꾸면 코드는 그대로 돌아가는데
   왜 문제인가? 이 노트북에서 그 차이를 무슨 수치로 보였는가?
4. L2 정규화를 하면 코사인 유사도 계산이 어떻게 달라지는가? 정규화하지 않은 벡터를
   내적 기준으로 검색하면 어떤 편향이 생기는가?
5. `model.encode()` 한 줄과 이 노트북의 수동 경로가 같다는 것을 어떻게 증명했는가?
   `atol=1e-5`를 쓴 이유는 무엇이며, 정확히 일치(`==`)를 요구하지 않는 까닭은?
6. 문서가 `max_seq_length`를 넘으면 무슨 일이 일어나는가? 그 사고가 운영 중에
   발견되기 어려운 이유는 무엇이고, 어떻게 탐지·방지할 것인가?
7. 10건 전수 코사인은 계산량이 얼마인가(곱셈 횟수 기준)? 문서가 100만 건이 되면
   이 방식의 무엇이 먼저 무너지고, 그래서 #6에서 무엇을 무엇과 대조하려는 것인가?